In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token GITHUB_TOKEN_{token_index + 1} of {len(tokens)}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step1_search_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
failed_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_failed_requests.csv"

# === Read input ===
df = pd.read_csv(input_path)

# === Output columns to add ===
df["rest_check_status"] = ""
df["rest_reason"] = ""

failed_repos = []

# === Define valid languages ===
valid_langs = ["Java", "Kotlin", "Dart"]

MAX_RETRIES = 3

# === REST validation loop ===
for i, row in df.iterrows():
    if row["Valid_Repo"] != "yes":
        df.at[i, "rest_check_status"] = "skip"
        df.at[i, "rest_reason"] = "invalid from step 1"
        continue

    repo = row["full_name"]
    url = f"https://api.github.com/repos/{repo}"

    retries = 0
    while retries < MAX_RETRIES:
        response = requests.get(url, headers=get_headers())
        if response.status_code == 403:
            print(f"⏳ Rate limit hit for {repo}. Retrying in 10s...")
            sleep(10)
            retries += 1
            continue
        break

    if response.status_code != 200:
        print(f"❌ Failed: {repo} - HTTP {response.status_code}")
        df.at[i, "rest_check_status"] = "reject"
        df.at[i, "rest_reason"] = f"HTTP {response.status_code}"
        df.at[i, "Valid_Repo"] = "no"
        failed_repos.append({"full_name": repo, "error": response.status_code})
        sleep(1)
        continue

    data = response.json()
    reasons = []

    if data.get("fork", True):
        reasons.append("fork")
    if data.get("archived", True):
        reasons.append("archived")
    if data.get("stargazers_count", 0) <= 50:
        reasons.append("low stars")
    if data.get("language") not in valid_langs:
        reasons.append("language mismatch")

    if reasons:
        df.at[i, "rest_check_status"] = "reject"
        df.at[i, "rest_reason"] = ", ".join(reasons)
        df.at[i, "Valid_Repo"] = "no"
    else:
        df.at[i, "rest_check_status"] = "pass"
        df.at[i, "rest_reason"] = ""

    if i % 100 == 0:
        print(f"✅ Checked {i + 1} repos...")

# === Save outputs ===
df.to_csv(output_path, index=False)
print(f"✅ Step 2 complete. Verified data saved to: {output_path}")

if failed_repos:
    pd.DataFrame(failed_repos).to_csv(failed_path, index=False)
    print(f"⚠️ Failed requests saved to: {failed_path}")
